### Goals

- load models trained with different setups
- with each model, predict the test set  
- compute metrics







In [1]:
import sys
import matplotlib.pyplot as plt
from IPython.display import clear_output
import json
import itertools 
import os 
import random
import matplotlib.pyplot as plt
import pprint
sys.path.append('../')
import tqdm
from pprint import pformat
import pickle

from sklearn.metrics import classification_report
import torch.nn as nn
import rasterio
import numpy as np
import torch
import torch.nn.functional as F
import utils
import pandas as pd
import geopandas as gpd

import model.dataloader as dataloader
from model.dataset import LABEL_MAPPING
from model.model import MultiModalDriverClassifier

import wandb

inverse_label_mapping = {value : key for key, value in LABEL_MAPPING.items()}

In [2]:
import sys
sys.path.append('../')
from evaluate import compute_and_log_metrics


### Examples of how to filter wandb runs:

example_filters = {


    "group" : "17/5 single-bi-trimodal",
    "config.image_encoder.use": True,
    "config.batch_size": {'$gt': 10}, #Greater than
    "config.batch_size": {'$lte': 10}, #Less than or equal
    
	"$and": [{
		'created_at': {
			"$lt": '2022-03-09T10',
			"$gt": '2022-03-07T17'
		}
	}]
    
}

In [3]:
# Define the filter criteria
filter_criteria = {

    "group" : "16/7 ISPRS review",

    #"config.seed" : 420,
    #"$and": [{
    #'created_at': {
    #        "$lt": '2024-09-26T00',
    #        "$gt": '2024-09-24T00'
    #    }
    #}]

    
}





In [4]:
models = utils.get_wandb_runs(filter_criteria)

print(f'models: {len(models)}')

models: 10


In [5]:
models = models[:5]

In [10]:
'''
In some cases, the wandb API filtering doesn't work;
If I get 0 models returned, I need to 
    1. remove the problematic filter
    2. get all the runs
    3. filter manually as below
'''
#Example:
#models = [model for model in models  if model['config']['seed'] == 422]  
#print(len(selected_models))
#models = selected_models

"\nIn some cases, the wandb API filtering doesn't work;\nIf I get 0 models returned, I need to \n    1. remove the problematic filter\n    2. get all the runs\n    3. filter manually as below\n"

In [7]:
print(len(models))

5


In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda'
n_classes = 11


In [9]:
CHECKPOINT_TYPE = 'last'

for i, model_dict in enumerate(tqdm.tqdm(models)):
    #if i < 16:
    #    continue
    #if i > 2:
    #    break
    
    #This is just to get rid of wandb objects that are un-picklable wtf
    model_dict['summary'] = None
    
    try:
        metrics, preds = utils.load_model_and_predict(model_dict, n_classes, device, 'val', CHECKPOINT_TYPE)
    except Exception as e:
        print(e)
        continue
    model_dict['predictions'] = preds
    model_dict['metrics'] = metrics
    
    #try:
    #    model_dict['summary']['_wandb'] = dict(model_dict['summary']['_wandb'])
    #    model_dict['summary']['val conf matrix'] = None
    #    model_dict['summary']['train conf matrix'] = None
    #except KeyError:
    #    pass
    

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:33<00:00,  6.80s/it]


In [26]:
import importlib
importlib.reload(utils)


<module 'utils' from '/home/jan/Documents/EPFL/project_2/notebooks/../utils.py'>

In [12]:
hyperparams_values = utils.find_keys_with_multiple_values([model['config'] for model in models])

In [13]:
hyperparams_values

{'seed': [421, 422, 423, 424, 425]}

In [14]:
labels = pd.read_csv("/home/jan/Documents/EPFL/project_2/data/labels/dataset_v5/labels_no_regrowth.csv")
labels = labels.loc[labels['split'] == 4].reset_index(drop=True) # Get test examples 

targets = labels['driver'].map(LABEL_MAPPING).to_numpy()


In [15]:
def get_models_with_config(_models, target_config):
    
    def has_key_value_pair(d, target):
        if isinstance(d, dict) and isinstance(target, dict):
            for key, value in target.items():
                if key in d:
                    if isinstance(value, dict) and isinstance(d[key], dict):
                        if not has_key_value_pair(d[key], value):
                            return False
                    else:
                        if d[key] != value:
                            return False
                else:
                    return False
            return True
        return d == target
    
    models_with_this_config_combination = []
    for model in _models:
        skip_model = False
        for key in target_config.keys():
            if not has_key_value_pair(model['config'][key],target_config[key]):
            #if model['config'][key] != target_config[key]:
                skip_model = True
                break
        if skip_model:
            continue
        models_with_this_config_combination.append(model)
    
    return models_with_this_config_combination

In [19]:
hyperparams_values

{'seed': [421, 422, 423, 424, 425]}

In [21]:
exclude_keys = []

combinations = utils.generate_combinations(hyperparams_values, exclude_keys)

In [22]:
combinations

[{'seed': 421}, {'seed': 422}, {'seed': 423}, {'seed': 424}, {'seed': 425}]

In [32]:
#This I need to do in case there is only 1 seed
#hyperparams_values['seed'] = [420]

In [40]:

performance_per_configuration = {}

for combo in combinations:
    
    metrics_per_random_seed = []
    combo_str = ' '.join(pformat(combo, width=1).split())
    #print(combo_str)

    #Compute metrics per each random seed separately
    for seed in hyperparams_values['seed']:
        combo['seed'] = seed 
        
        #Get all models that have the configuration setup 
        model_with_this_config = get_models_with_config(models, combo)
        if not model_with_this_config:
            continue

            
        assert len(model_with_this_config) == 1, f"there should be 1 model with a config, found {len(model_with_this_config)}"
        model_with_this_config = model_with_this_config[0]
        if not 'metrics' in model_with_this_config.keys():
            continue
            
            
        metrics_per_random_seed.append(pd.DataFrame(model_with_this_config['metrics']))
        print(model_with_this_config['metrics']['accuracy'])


    if not metrics_per_random_seed:
        print("no metrics for: ", combo)
        continue
    
    # Concatenate DataFrames along a new axis
    concat_df = pd.concat(metrics_per_random_seed, keys=range(len(metrics_per_random_seed)))

    # Compute the mean along the new axis
    mean_df = concat_df.groupby(level=1).mean()

    # Compute the standard deviation along the new axis
    std_df = concat_df.groupby(level=1).std()   
    
    

    performance_per_configuration[combo_str] = {}
    performance_per_configuration[combo_str]['mean'] = mean_df
    performance_per_configuration[combo_str]['std'] = std_df



        

0.8511041009463722
0.8580441640378549
0.8250262881177708
0.861198738170347
0.8513144058885384
0.8511041009463722
0.8580441640378549
0.8250262881177708
0.861198738170347
0.8513144058885384
0.8511041009463722
0.8580441640378549
0.8250262881177708
0.861198738170347
0.8513144058885384
0.8511041009463722
0.8580441640378549
0.8250262881177708
0.861198738170347
0.8513144058885384
0.8511041009463722
0.8580441640378549
0.8250262881177708
0.861198738170347
0.8513144058885384


In [39]:
model_with_this_config['metrics']['accuracy']

0.8511041009463722

In [33]:
for seed in [421,422,423,424,425]:
    print(seed)
    print(performance_per_configuration[f"{{'seed': {seed}}}"]['mean']['accuracy']['f1-score'])

421
0.8493375394321767
422
0.8493375394321767
423
0.8493375394321767
424
0.8493375394321767
425
0.8493375394321767


In [35]:
print(performance_per_configuration["{'seed': 422}"]['mean']['accuracy']['f1-score'])

0.8493375394321767
